In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    import torch; v = re.match(r"[0-9]{1,}\.[0-9]{1,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.33.post1" if v=="2.9" else "0.0.32.post2" if v=="2.8" else "0.0.29.post3")
    !pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2
!pip install -qq -U evaluate rouge_score

In [ ]:
import unsloth
from unsloth import FastLanguageModel
import torch

import os
os.environ["UNSLOTH_RETURN_LOGITS"] = "1"
max_seq_length = 1024

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/DeepSeek-R1-Distill-Qwen-1.5B-bnb-4bit",
    max_seq_length = max_seq_length,   # Define context length
    load_in_4bit = True,     # 4bit uses much less memory
    load_in_8bit = False,    # A bit more accurate, uses 2x memory
    full_finetuning = False, # We have full finetuning now!
    # token = "hf_...",      # Add your token if using a gated model
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


2026-03-24 16:14:56.638118: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774368896.806302      99 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774368896.856994      99 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774368897.235312      99 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774368897.235346      99 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774368897.235349      99 computation_placer.cc:177] computation placer alr

In [63]:
# Add LoRA adapters to the model
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,           # LoRA rank (higher rank = more parameters, potentially better fit but more memory)
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", # Target attention and MLP layers
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,  # Scaling factor (often set to r or 2*r)
    lora_dropout = 0, # Dropout probability for LoRA layers
    bias = "none",    # Fine-tuning bias terms ('none' is often optimal)
    # Use Unsloth's gradient checkpointing for memory saving
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False, # Rank Stable LoRA (optional)
    loftq_config = None, # LoftQ initialization (optional)
)

In [64]:
# ─────────────────────────────────────────────────────────────────────────────
# STEP 2 — LOAD + COMBINE DATASETS
# ─────────────────────────────────────────────────────────────────────────────
print("[ 2/5 ] Loading datasets")
from datasets import load_dataset  , concatenate_datasets
import random
import gc
SEED = 42

SEED           = 42
TARGET_TOTAL   = 40000        # 20k is sufficient — quality >> quantity for SLMs

WEIGHTS = {
    "instruction":   0.35,
    "summarization": 0.30,
    "extraction":    0.20,
    "decomposition": 0.15,
}

ds1 = load_dataset("Rohit-Katkar2003/Instruction-dataset",   split="train")
ds2 = load_dataset("Rohit-Katkar2003/summarization-dataset", split="train")
ds3 = load_dataset("Rohit-Katkar2003/Extractional-dataset",  split="train")
ds4 = load_dataset("Rohit-Katkar2003/decomposition-dataset", split="train")

print(f"    D1={len(ds1):,}  D2={len(ds2):,}  D3={len(ds3):,}  D4={len(ds4):,}")

random.seed(SEED)
splits = []
for name, ds, w in [
    ("instruction",   ds1, WEIGHTS["instruction"]),
    ("summarization", ds2, WEIGHTS["summarization"]),
    ("extraction",    ds3, WEIGHTS["extraction"]),
    ("decomposition", ds4, WEIGHTS["decomposition"]),
]:
    target_n = int(TARGET_TOTAL * w)
    if len(ds) >= target_n:
        idx = random.sample(range(len(ds)), target_n)
    else:
        repeats = (target_n // len(ds)) + 1
        idx     = (list(range(len(ds))) * repeats)[:target_n]
        random.shuffle(idx)
        print(f"    Upsampled {name}: {len(ds)} → {target_n}")
    splits.append(ds.select(idx))

combined = concatenate_datasets(splits).shuffle(seed=SEED)
del ds1, ds2, ds3, ds4
gc.collect()
print(f"[ 2/5 ] Combined: {len(combined):,} samples ✓")

[ 2/5 ] Loading datasets
    D1=71,179  D2=100,000  D3=100,000  D4=500
    Upsampled decomposition: 500 → 6000
[ 2/5 ] Combined: 40,000 samples ✓


In [65]:
combined[1]

{'dataset': 'arxiv-abstracts',
 'task': 'summarization',
 'system': 'You are a research summarization assistant. Your summaries are accurate, concise, and preserve all key facts.',
 'messages': [{'role': 'user',
   'content': 'Extract the key findings and main points from this text:\n\n  Liesegang patterns emerge from precipitation processes and may be used to\nbuild bulk structures at submicron lengthscales. Thus they have significant\npotential for technological applications provided adequate methods of control\ncan be devised. Here we describe a simple, physically realizable\npattern-control based on the notion of driven precipitation, meaning that the\nphase-separation is governed by a guiding field such as, for example, a\ntemperature or a pH field. The phase-separation is modeled through a\nnon-autonomous Cahn-Hilliard equation whose spinodal is determined by the\nevolving guiding field. Control over the dynamics of the spinodal gives control\nover the velocity of the instability

In [66]:
def format_chat(example):
    system = example["system"]
    messages = example["messages"]

    text = f"<|system|>\n{system}\n"

    for msg in messages:
        if msg["role"] == "user":
            text += f"<|user|>\n{msg['content']}\n"
        elif msg["role"] == "assistant":
            text += f"<|assistant|>\n{msg['content']}\n"

    return {"text": text}

train_dataset = combined.map(format_chat)

In [67]:
split_dataset = train_dataset.train_test_split(test_size=0.15, seed=3407)
# Access the train and validation sets
train_dataset = split_dataset["train"]
valid_dataset = split_dataset["test"]

print("Train size:", len(train_dataset))
print("Valid size:", len(valid_dataset))
print(train_dataset[0])

Train size: 34000
Valid size: 6000
{'dataset': 'arxiv-abstracts', 'task': 'summarization', 'system': 'You are a research summarization assistant. Your summaries are accurate, concise, and preserve all key facts.', 'messages': [{'role': 'user', 'content': "Summarize the following text concisely:\n\n  Node betweenness has been studied recently by a number of authors, but until\nnow less attention has been paid to edge betweenness. In this paper, we present\nan exact analytic study of edge betweenness in evolving scale-free and\nnon-scale-free trees. We aim at the probability distribution of edge\nbetweenness under the condition that a local property, the in-degree of the\n``younger'' node of a randomly selected edge, is known. En route to the\nconditional distribution of edge betweenness the exact joint distribution of\ncluster size and in-degree, and its one dimensional marginal distributions have\nbeen presented in the paper as well. From the derived probability distributions\nthe expe

In [68]:
print("<|assistant|>" in train_dataset[0]["text"])

True


In [69]:
import numpy as np
from transformers import EvalPrediction
import evaluate

# Load metrics
rouge = evaluate.load("rouge")
bleu = evaluate.load("bleu")

In [70]:
def preprocess_logits_for_metrics(logits, labels):
    """Returns predicted token IDs (argmax) for metrics calculation"""
    if isinstance(logits, tuple):
        logits = logits[0]  # Unpack if needed
    return logits.argmax(dim=-1)

def compute_metrics(eval_preds: EvalPrediction):
    """Compute ROUGE, BLEU, AND token-level accuracy"""
    preds, labels = eval_preds

    # --- Token-Level Accuracy Calculation ---
    # Flatten all predictions/labels (ignore padding tokens)
    mask = labels != -100  # Only compare non-ignored tokens
    preds_flat = preds[mask].flatten()
    labels_flat = labels[mask].flatten()

    accuracy = (preds_flat == labels_flat).mean()

    # --- Text Generation Metrics ---
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Post-process text
    decoded_preds = [pred.strip() for pred in decoded_preds]
    decoded_labels = [[label.strip()] for label in decoded_labels]

    rouge_results = rouge.compute(
        predictions=decoded_preds,
        references=decoded_labels,
        use_stemmer=True
    )
    bleu_results = bleu.compute(
        predictions=decoded_preds,
        references=decoded_labels
    )

    return {
        "accuracy": float(accuracy),  # Token-level exact match
        "rouge1": rouge_results["rouge1"],
        "rouge2": rouge_results["rouge2"],
        "rougeL": rouge_results["rougeL"],
        "bleu": bleu_results["bleu"],
    }

def compute_metricsR(eval_preds: EvalPrediction):
    """Compute ROUGE, BLEU, AND token-level accuracy"""
    preds, labels = eval_preds

    # --- Token-Level Accuracy Calculation ---
    # Flatten all predictions/labels (ignore padding tokens)
    mask = labels != -100  # Only compare non-ignored tokens
    preds_flat = preds[mask].flatten()
    labels_flat = labels[mask].flatten()

    accuracy = (preds_flat == labels_flat).mean()

    # --- Text Generation Metrics ---
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Post-process text
    decoded_preds = [pred.strip() for pred in decoded_preds]
    decoded_labels = [[label.strip()] for label in decoded_labels]

    rouge_results = rouge.compute(
        predictions=decoded_preds,
        references=decoded_labels,
        use_stemmer=True
    )
    bleu_results = bleu.compute(
        predictions=decoded_preds,
        references=decoded_labels
    )

    return {
        "accuracy": float(accuracy),  # Token-level exact match
        "rouge1": rouge_results["rouge1"],
        "rouge2": rouge_results["rouge2"],
        "rougeL": rouge_results["rougeL"],
        "bleu": bleu_results["bleu"],
    }

In [71]:
type(train_dataset)

datasets.arrow_dataset.Dataset

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=None,

    args=SFTConfig(
        dataset_text_field="text",

        # 🔥 BATCH (IMPORTANT)
        per_device_train_batch_size=4,
        gradient_accumulation_steps=8,   # effective batch = 32

        # 🔥 TRAINING LENGTH
        num_train_epochs=2,
        # max_steps=100,                    # REMOVE 100 step limit

        # 🔥 LEARNING RATE (VERY IMPORTANT)
        learning_rate=2e-4,              # best for LoRA
        warmup_ratio=0.05,               # better than fixed steps

        # 🔥 OPTIMIZATION
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",      # better than linear

        # 🔥 STABILITY
        max_grad_norm=1.0,

        # 🔥 PERFORMANCE
        logging_steps=10,
        dataloader_num_workers=2,
        dataloader_pin_memory=True,

        # 🔥 PRECISION
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),

        # 🔥 MISC
        seed=3407,
        report_to="none",
    ),

    # 🔥 HUGE IMPACT (DON'T SKIP)
    packing=True,
)

In [73]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = Tesla T4. Max memory = 14.563 GB.
6.633 GB of memory reserved.


In [ ]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 34,000 | Num Epochs = 1 | Total steps = 1,063
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 8 x 1) = 32
 "-____-"     Trainable parameters = 18,464,768 of 1,795,552,768 (1.03% trained)


Step,Training Loss
10,3.320500
20,3.113600
30,2.783400
40,2.474600
50,2.329100
60,2.121100
70,2.084800
80,2.020700
90,2.014800
100,2.015800


In [45]:
# Start training
print("Starting training...")
trainer_stats = trainer.train()
print("Training finished.")

Starting training...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 33,934 | Num Epochs = 2 | Total steps = 2,122
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 8 x 1) = 32
 "-____-"     Trainable parameters = 18,464,768 of 1,795,552,768 (1.03% trained)


Step,Training Loss
20,2.081000
40,1.505900


KeyboardInterrupt: 